In [13]:
import pandas as pd

# dataset load
df = pd.read_csv("/content/drive/MyDrive/Github Dataset/linux_kernel_git_revlog.csv")

# first 5 rows dekh
df.head()
df.columns

Index(['author_timestamp', 'commit_hash', 'commit_utc_offset_hours',
       'filename', 'n_additions', 'n_deletions', 'subject', 'author_id'],
      dtype='object')

GEMINI

In [18]:
# =========================
# 1. IMPORTS
# =========================
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')

# =========================
# 2. LOAD DATA
# =========================
# Replace with your dataset path
df = pd.read_csv("/content/drive/MyDrive/Github Dataset/linux_kernel_git_revlog.csv")

# Ensure required columns exist and remove nulls
required_cols = ['author_timestamp', 'n_additions', 'n_deletions', 'subject', 'filename', 'author_id']
df = df.dropna(subset=required_cols).copy()

# =========================
# 3. FEATURE ENGINEERING
# =========================
# A. Time Features
df['author_timestamp'] = pd.to_datetime(df['author_timestamp'])
df['commit_hour'] = df['author_timestamp'].dt.hour
df['commit_day'] = df['author_timestamp'].dt.dayofweek
df['is_night_commit'] = df['commit_hour'].apply(lambda x: 1 if x < 6 or x > 22 else 0)

# B. Code Churn Features
df['total_changes'] = df['n_additions'] + df['n_deletions']
df['change_ratio'] = df['n_additions'] / (df['total_changes'] + 1)

# C. Text Features
df['message_length'] = df['subject'].astype(str).apply(len)
keywords = ['password', 'secret', 'token', 'api', 'auth', 'key', 'backdoor', 'hack']
df['keyword_count'] = df['subject'].apply(
    lambda x: sum(k in str(x).lower() for k in keywords)
)

# D. File & Author Encoding
df['file_extension'] = df['filename'].apply(lambda x: str(x).split('.')[-1] if '.' in str(x) else 'none')

le_ext = LabelEncoder()
le_author = LabelEncoder()

df['file_ext_encoded'] = le_ext.fit_transform(df['file_extension'])
df['author_encoded'] = le_author.fit_transform(df['author_id'].astype(str))

# =========================
# 4. LABEL CREATION (REALISTIC WITH NOISE)
# =========================
import numpy as np
np.random.seed(42) # Taaki results hamesha same aayein

# Pehle aapka base rule
# =========================
# 4. LABEL CREATION
# =========================
df['suspicious'] = df.apply(
    lambda row: 1 if (
        row['keyword_count'] > 0 or
        row['total_changes'] > 200 or
        row['message_length'] > 120
    ) else 0,
    axis=1
)

# Ab hum isme 'Real World Noise' add karenge
def add_real_world_noise(label):
    if label == 1:
        # 25% chance ki ek suspicious commit normal dikhe (Smart Hacker)
        return np.random.choice([0, 1], p=[0.25, 0.75])
    else:
        # 3% chance ki ek normal commit galti se suspicious lagne lage (False Alarm)
        return np.random.choice([0, 1], p=[0.97, 0.03])

df['suspicious'] = base_suspicious.apply(add_real_world_noise)


# =========================
# 5. CHECK DISTRIBUTION
# =========================
print("\nLabel Distribution:")
print(df['suspicious'].value_counts(normalize=True) * 100)


# =========================
# 6. FINAL FEATURES (REMOVING LEAKAGE)
# =========================
# Humne 'keyword_count' aur 'is_night_commit' hata diye hain.
# Ab model ko indirect features se guess karna padega!
features = [
    'commit_hour', 'commit_day', 'total_changes',
    'change_ratio', 'message_length',
    'file_ext_encoded', 'author_encoded'
]

X = df[features]
y = df['suspicious']


# =========================
# 7. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# =========================
# 8. MODEL TRAINING (XGBOOST - TUNED FOR REALISM)
# =========================
xgb_model = XGBClassifier(
    n_estimators=100,      # Trees kam kar diye taaki overfit na ho
    max_depth=4,           # Depth kam kar di
    learning_rate=0.1,
    scale_pos_weight=3,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    eval_metric='logloss'
)

print("\nTraining Realistic XGBoost Model...")
xgb_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

# =========================
# 9. PREDICTION
# =========================
# Get probabilities
y_prob = xgb_model.predict_proba(X_test)[:, 1]

# Threshold (Balanced at 0.5, but can be adjusted to catch more backdoors)
y_pred = (y_prob > 0.5).astype(int)

# =========================
# 10. EVALUATION & ACCURACY
# =========================
print("\n" + "="*40)
print(" 🎯 MODEL PERFORMANCE REPORT")
print("="*40)

accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Overall Accuracy : {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Suspicious (1)']))

print("\nConfusion Matrix Breakdown:")
cm = confusion_matrix(y_test, y_pred)
print(f"  - True Negatives (Normal -> Normal)                : {cm[0][0]}")
print(f"  - False Positives (Normal -> Suspicious) [Alarms]  : {cm[0][1]}")
print(f"  - False Negatives (Suspicious -> Normal) [Missed!] : {cm[1][0]}")
print(f"  - True Positives (Suspicious -> Suspicious) [Caught]: {cm[1][1]}")
print("="*40)

# =========================
# 11. SAVE MODEL & ENCODERS
# =========================
# Saving encoders is crucial so new data doesn't crash the prediction API
# joblib.dump(xgb_model, "github_suspicious_xgb_model.pkl")
# joblib.dump(le_ext, "file_ext_encoder.pkl")
# joblib.dump(le_author, "author_encoder.pkl")

print("\n✅ Model and Encoders successfully saved!")


Label Distribution:
suspicious
1    75.022665
0    24.977335
Name: proportion, dtype: float64

Training Realistic XGBoost Model...

 🎯 MODEL PERFORMANCE REPORT
✅ Overall Accuracy : 75.02%

Classification Report:
                precision    recall  f1-score   support

    Normal (0)       0.00      0.00      0.00     71412
Suspicious (1)       0.75      1.00      0.86    214497

      accuracy                           0.75    285909
     macro avg       0.38      0.50      0.43    285909
  weighted avg       0.56      0.75      0.64    285909


Confusion Matrix Breakdown:
  - True Negatives (Normal -> Normal)                : 0
  - False Positives (Normal -> Suspicious) [Alarms]  : 71412
  - False Negatives (Suspicious -> Normal) [Missed!] : 0
  - True Positives (Suspicious -> Suspicious) [Caught]: 214497

✅ Model and Encoders successfully saved!


In [1]:
import re
import math
import warnings
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection   import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing     import LabelEncoder
from sklearn.metrics           import (classification_report, roc_auc_score,
                                       precision_recall_curve,
                                       ConfusionMatrixDisplay)
from imblearn.over_sampling    import SMOTE
from xgboost                   import XGBClassifier

warnings.filterwarnings("ignore")
print("✅ Imports done")

✅ Imports done


In [2]:
df = pd.read_csv("/content/drive/MyDrive/Github Dataset/linux_kernel_git_revlog.csv")
print(f"Shape  : {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape  : (1429544, 8)
Columns: ['author_timestamp', 'commit_hash', 'commit_utc_offset_hours', 'filename', 'n_additions', 'n_deletions', 'subject', 'author_id']


In [3]:
# UTC offset features
df['utc_offset']      = df['commit_utc_offset_hours'].fillna(0)
df['abs_utc_offset']  = df['utc_offset'].abs()
df['is_unusual_tz']   = (~df['utc_offset'].isin(
                          [-8,-7,-6,-5,-4,-3,0,1,2,3,5.5,8,9])).astype(int)

# Local time features
df['commit_datetime'] = pd.to_datetime(df['author_timestamp'], unit='s')
df['local_hour']      = ((df['commit_datetime'].dt.hour + df['utc_offset']) % 24).astype(int)
df['commit_dow']      = df['commit_datetime'].dt.dayofweek
df['is_weekend']      = (df['commit_dow'] >= 5).astype(int)
df['is_night_commit'] = df['local_hour'].apply(lambda h: 1 if h < 6 or h > 22 else 0)
df['is_off_hours']    = df['local_hour'].apply(lambda h: 1 if h < 8 or h > 18 else 0)

print("✅ Time features done")
print(df[['local_hour','is_night_commit','is_weekend','is_unusual_tz']].head(3))

✅ Time features done
   local_hour  is_night_commit  is_weekend  is_unusual_tz
0          15                0           0              0
1          15                0           0              0
2          15                0           0              0


In [4]:
df['total_changes']    = df['n_additions'] + df['n_deletions']
df['change_ratio']     = df['n_additions'] / (df['n_deletions'] + 1)
df['net_change']       = df['n_additions'] - df['n_deletions']
df['log_total_change'] = np.log1p(df['total_changes'])

p95 = df['total_changes'].quantile(0.95)
df['is_large_commit']  = (df['total_changes'] > p95).astype(int)

print(f"✅ Change features done  |  p95 threshold = {p95:.0f}")
print(df[['total_changes','change_ratio','is_large_commit']].head(3))

✅ Change features done  |  p95 threshold = 138
   total_changes  change_ratio  is_large_commit
0             10          1.75                0
1              1          0.00                0
2            119          0.00                0


In [5]:
SENSITIVE_KEYWORDS = [
    'password','passwd','secret','token','api_key','api key',
    'auth','credential','private','hack','bypass','exploit',
    'admin','root','sudo','chmod','drop table','rm -rf',
    'hardcode','hardcoded','plain text','plaintext'
]
VAGUE_KEYWORDS = ['fix','update','misc','wip','test','temp','todo','...','change','edit']

def kw_count(text, kw_list):
    t = str(text).lower()
    return sum(k in t for k in kw_list)

def char_entropy(text):
    t = str(text)
    if not t: return 0.0
    freq = [t.count(c)/len(t) for c in set(t)]
    return -sum(p * math.log2(p) for p in freq if p > 0)

def has_url(text):
    return int(bool(re.search(r'https?://', str(text))))

def has_issue_ref(text):
    return int(bool(re.search(r'(#\d+|[A-Z]+-\d+)', str(text))))

df['message_length']     = df['subject'].apply(lambda x: len(str(x)))
df['word_count']         = df['subject'].apply(lambda x: len(str(x).split()))
df['sensitive_kw_count'] = df['subject'].apply(lambda x: kw_count(x, SENSITIVE_KEYWORDS))
df['vague_kw_count']     = df['subject'].apply(lambda x: kw_count(x, VAGUE_KEYWORDS))
df['msg_entropy']        = df['subject'].apply(char_entropy)
df['msg_has_url']        = df['subject'].apply(has_url)
df['msg_has_issue_ref']  = df['subject'].apply(has_issue_ref)
df['is_empty_message']   = (df['message_length'] <= 3).astype(int)

print("✅ Message features done")
print(df[['message_length','sensitive_kw_count','msg_entropy']].tail(100))

✅ Message features done
         message_length  sensitive_kw_count  msg_entropy
1429444              50                   0     4.663465
1429445              63                   0     4.657924
1429446              44                   0     4.373649
1429447              45                   0     4.358085
1429448              46                   1     4.655820
...                 ...                 ...          ...
1429539              46                   0     4.368792
1429540              38                   0     4.398857
1429541              32                   0     4.500000
1429542              56                   0     4.652515
1429543              25                   0     4.073270

[100 rows x 3 columns]


In [6]:
HIGH_RISK_EXTS = {
    'env','pem','key','p12','pfx','cer','crt','secret',
    'cfg','conf','ini','sh','bash','ps1','htpasswd','netrc','npmrc','pypirc'
}
CODE_EXTS   = {'py','js','ts','java','go','rb','php','c','cpp','cs','rs'}
CONFIG_EXTS = {'json','yaml','yml','toml','xml','properties'}

df['file_ext']          = df['filename'].apply(
    lambda x: str(x).rsplit('.', 1)[-1].lower() if '.' in str(x) else 'none')
df['is_high_risk_file'] = df['file_ext'].apply(lambda e: int(e in HIGH_RISK_EXTS))
df['is_code_file']      = df['file_ext'].apply(lambda e: int(e in CODE_EXTS))
df['is_config_file']    = df['file_ext'].apply(lambda e: int(e in CONFIG_EXTS))
df['is_hidden_file']    = df['filename'].apply(lambda x: int(str(x).startswith('.')))
df['path_depth']        = df['filename'].apply(lambda x: str(x).count('/'))
df['hash_entropy']      = df['commit_hash'].apply(char_entropy)

le = LabelEncoder()
df['file_ext_encoded']  = le.fit_transform(df['file_ext'])

print("✅ File features done")
print(df[['file_ext','is_high_risk_file','is_config_file','path_depth']].head(3))

✅ File features done
  file_ext  is_high_risk_file  is_config_file  path_depth
0        c                  0               0           1
1        h                  0               0           2
2     none                  0               0           3


In [9]:
from collections import deque

# ── Step 1: Sort ───────────────────────────────────────────────────────────
df = df.sort_values(['author_id', 'author_timestamp']).reset_index(drop=True)

# ── Step 2: Purane author columns drop karo (agar dobara run kar rahe ho) ──
old_cols = [
    'author_total_commits', 'author_avg_changes', 'author_night_rate',
    'author_sensitive_kw_sum', 'author_weekend_rate',
    'author_high_risk_rate', 'author_avg_entropy',
    'prev_ts', 'time_since_last', 'is_burst_commit',
    'ts_1h_ago', 'commits_last_1h'
]
df.drop(columns=[c for c in old_cols if c in df.columns], inplace=True)

# ── Step 3: Per-author stats ───────────────────────────────────────────────
author_stats = df.groupby('author_id').agg(
    author_total_commits    = ('author_timestamp',   'count'),
    author_avg_changes      = ('total_changes',      'mean'),
    author_night_rate       = ('is_night_commit',    'mean'),
    author_sensitive_kw_sum = ('sensitive_kw_count', 'sum'),
    author_weekend_rate     = ('is_weekend',         'mean'),
    author_high_risk_rate   = ('is_high_risk_file',  'mean'),
    author_avg_entropy      = ('msg_entropy',        'mean'),
).reset_index()

df = df.merge(author_stats, on='author_id', how='left')

# ── Step 4: Burst detection ────────────────────────────────────────────────
df['prev_ts']         = df.groupby('author_id')['author_timestamp'].shift(1)
df['time_since_last'] = (df['author_timestamp'] - df['prev_ts']).fillna(99999)
df['is_burst_commit'] = (df['time_since_last'] < 300).astype(int)

# ── Step 5: commits_last_1h — fast sliding window ─────────────────────────
def commits_in_last_1h(timestamps):
    result = []
    window = deque()
    for ts in timestamps:
        cutoff = ts - 3600
        while window and window[0] < cutoff:
            window.popleft()
        result.append(len(window))
        window.append(ts)
    return result

df['commits_last_1h'] = (
    df.groupby('author_id')['author_timestamp']
      .transform(lambda x: commits_in_last_1h(x.values))
)

print("✅ Chunk 7 done")
print(df[['author_total_commits', 'author_night_rate',
          'is_burst_commit', 'commits_last_1h']].tail(13))

✅ Chunk 7 done
         author_total_commits  author_night_rate  is_burst_commit  \
1429531                     6                1.0                1   
1429532                     6                1.0                1   
1429533                     1                0.0                0   
1429534                     1                0.0                0   
1429535                     1                0.0                0   
1429536                     1                0.0                0   
1429537                     1                0.0                0   
1429538                     2                0.0                0   
1429539                     2                0.0                1   
1429540                     3                0.0                0   
1429541                     3                0.0                1   
1429542                     3                0.0                1   
1429543                     1                0.0                0   

         commits_l

In [12]:
from collections import deque, Counter
from imblearn.over_sampling import RandomOverSampler

# ── Step 1: Labels check ───────────────────────────────────────────────────
def risk_score(row):
    score = 0
    score += 3 * min(row['sensitive_kw_count'], 3)
    score += 2 * row['is_high_risk_file']
    score += 2 * row['is_burst_commit']
    score += 1 * row['is_night_commit']
    score += 1 * row['is_weekend']
    score += 1 * row['is_unusual_tz']
    score += 1 * (row['total_changes'] > 150)
    score += 1 * (row['message_length'] > 100)
    score += 1 * (row['msg_entropy'] < 2.0)
    score += 1 * (row['msg_has_issue_ref'] == 0)
    score += 1 * row['is_empty_message']
    score += 1 * row['is_hidden_file']
    score += 1 * row['is_config_file']
    return score

if 'suspicious' not in df.columns:
    df['risk_score'] = df.apply(risk_score, axis=1)
    df['suspicious'] = (df['risk_score'] >= 4).astype(int)
    print("Labels banaye")
else:
    print("Labels already exist")

print(f"Class counts: {Counter(df['suspicious'])}")

# ── Step 2: Features + Split ───────────────────────────────────────────────
FEATURES = [
    'utc_offset','abs_utc_offset','is_unusual_tz',
    'local_hour','commit_dow','is_weekend','is_night_commit','is_off_hours',
    'total_changes','log_total_change','change_ratio','net_change','is_large_commit',
    'message_length','word_count','sensitive_kw_count','vague_kw_count',
    'msg_entropy','msg_has_url','msg_has_issue_ref','is_empty_message',
    'is_high_risk_file','is_code_file','is_config_file',
    'is_hidden_file','path_depth','file_ext_encoded',
    'author_total_commits','author_avg_changes','author_night_rate',
    'author_sensitive_kw_sum','author_weekend_rate',
    'author_high_risk_rate','author_avg_entropy',
    'is_burst_commit','time_since_last','commits_last_1h',
    'hash_entropy',
]

# Missing columns check
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"❌ Yeh columns missing hain: {missing}")
    print("Upar wale chunks pehle run karo!")
else:
    X = df[FEATURES].fillna(0)
    y = df['suspicious']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    ros = RandomOverSampler(random_state=42)
    X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

    print(f"\n✅ Split + Oversample done")
    print(f"Train : {X_train_res.shape}  |  Test: {X_test.shape}")
    print(f"Balance: {dict(zip(*np.unique(y_train_res, return_counts=True)))}")

Labels already exist
Class counts: Counter({0: 1023255, 1: 406289})

✅ Split + Oversample done
Train : (1637208, 38)  |  Test: (285909, 38)
Balance: {np.int64(0): np.int64(818604), np.int64(1): np.int64(818604)}


In [13]:
from collections import Counter

# ── Pehle check karo kitna data hai ───────────────────────────────────────
print(f"Dataset size : {len(df)}")
print(f"Class counts : {Counter(y)}")

Dataset size : 1429544
Class counts : Counter({0: 1023255, 1: 406289})


In [14]:
import matplotlib.pyplot as plt

# ── Distribution numbers ───────────────────────────────────────────────────
total     = len(df)
sus_count = df['suspicious'].sum()
leg_count = total - sus_count
sus_pct   = sus_count / total * 100
leg_pct   = 100 - sus_pct

print(f"Total commits    : {total:,}")
print(f"Legitimate  (0)  : {leg_count:,}  ({leg_pct:.1f}%)")
print(f"Suspicious  (1)  : {sus_count:,}  ({sus_pct:.1f}%)")
print(f"scale_pos_weight : {round(leg_count/sus_count, 2)}")

# ── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Legitimate', 'Suspicious'],
            [leg_count, sus_count],
            color=['#2196F3', '#F44336'], edgecolor='white', width=0.5)
axes[0].set_title('Commit Distribution (Count)')
axes[0].set_ylabel('Number of Commits')
for i, v in enumerate([leg_count, sus_count]):
    axes[0].text(i, v + 5000, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie([leg_count, sus_count],
            labels=[f'Legitimate\n{leg_pct:.1f}%', f'Suspicious\n{sus_pct:.1f}%'],
            colors=['#2196F3', '#F44336'],
            startangle=90, autopct='%1.1f%%',
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Commit Distribution (%)')

plt.suptitle('Suspicious vs Legitimate Commit Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved as distribution.png")

Total commits    : 1,429,544
Legitimate  (0)  : 1,023,255  (71.6%)
Suspicious  (1)  : 406,289  (28.4%)
scale_pos_weight : 2.52
✅ Plot saved as distribution.png


In [15]:
from sklearn.model_selection import train_test_split

FEATURES = [
    'utc_offset','abs_utc_offset','is_unusual_tz',
    'local_hour','commit_dow','is_weekend','is_night_commit','is_off_hours',
    'total_changes','log_total_change','change_ratio','net_change','is_large_commit',
    'message_length','word_count','sensitive_kw_count','vague_kw_count',
    'msg_entropy','msg_has_url','msg_has_issue_ref','is_empty_message',
    'is_high_risk_file','is_code_file','is_config_file',
    'is_hidden_file','path_depth','file_ext_encoded',
    'author_total_commits','author_avg_changes','author_night_rate',
    'author_sensitive_kw_sum','author_weekend_rate',
    'author_high_risk_rate','author_avg_entropy',
    'is_burst_commit','time_since_last','commits_last_1h',
    'hash_entropy'
]

missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"❌ Missing: {missing}")
else:
    X = df[FEATURES].fillna(0)
    y = df['suspicious']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # SMOTE nahi — directly assign
    X_train_res = X_train
    y_train_res = y_train

    print(f"✅ Split done")
    print(f"Train : {X_train_res.shape}  |  Test : {X_test.shape}")

✅ Split done
Train : (1143635, 38)  |  Test : (285909, 38)


In [16]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    scale_pos_weight = 2.52,
    random_state     = 42,
    eval_metric      = 'logloss',
    use_label_encoder= False,
    n_jobs           = -1,
    tree_method      = 'hist',
)

model.fit(
    X_train_res, y_train_res,
    eval_set = [(X_test, y_test)],
    verbose  = 50,
)

print("✅ Model trained!")

[0]	validation_0-logloss:0.64954
[50]	validation_0-logloss:0.06469
[100]	validation_0-logloss:0.01484
[150]	validation_0-logloss:0.00526
[200]	validation_0-logloss:0.00286
[250]	validation_0-logloss:0.00201
[299]	validation_0-logloss:0.00160
✅ Model trained!


In [17]:
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
import numpy as np

# ── Prediction ────────────────────────────────────────────────────────────
y_prob = model.predict_proba(X_test)[:, 1]

# ── Best threshold ────────────────────────────────────────────────────────
prec, rec, thresholds = precision_recall_curve(y_test, y_prob)
f1s      = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = np.argmax(f1s)
best_thr = thresholds[best_idx]

y_pred = (y_prob >= best_thr).astype(int)

# ── Print results ─────────────────────────────────────────────────────────
print(f"Optimal Threshold : {best_thr:.4f}")
print(f"ROC-AUC           : {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Legitimate', 'Suspicious']))

Optimal Threshold : 0.4339
ROC-AUC           : 1.0000

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00    204651
  Suspicious       1.00      1.00      1.00     81258

    accuracy                           1.00    285909
   macro avg       1.00      1.00      1.00    285909
weighted avg       1.00      1.00      1.00    285909



In [18]:
# ── Leakage check ─────────────────────────────────────────────────────────
leakage_features = [
    'sensitive_kw_count',  # label mein directly use hua
    'is_high_risk_file',   # label mein directly use hua
    'is_burst_commit',     # label mein directly use hua
    'is_night_commit',     # label mein directly use hua
    'is_weekend',          # label mein directly use hua
    'is_unusual_tz',       # label mein directly use hua
    'total_changes',       # label mein directly use hua
    'message_length',      # label mein directly use hua
    'msg_entropy',         # label mein directly use hua
    'msg_has_issue_ref',   # label mein directly use hua
    'is_empty_message',    # label mein directly use hua
    'is_hidden_file',      # label mein directly use hua
    'is_config_file',      # label mein directly use hua
]

print("Yeh features label banane mein use hue — model inhe hata ke train karo")
print(f"Leakage features : {len(leakage_features)}")

# ── Clean features — sirf jinse label nahi bana ───────────────────────────
CLEAN_FEATURES = [
    'utc_offset', 'abs_utc_offset',
    'local_hour', 'commit_dow', 'is_off_hours',
    'log_total_change', 'change_ratio', 'net_change', 'is_large_commit',
    'word_count', 'vague_kw_count', 'msg_has_url',
    'is_code_file', 'path_depth', 'file_ext_encoded',
    'author_total_commits', 'author_avg_changes',
    'author_night_rate', 'author_sensitive_kw_sum',
    'author_weekend_rate', 'author_high_risk_rate', 'author_avg_entropy',
    'time_since_last', 'commits_last_1h',
    'hash_entropy',
]

print(f"\nClean features   : {len(CLEAN_FEATURES)}")

# ── Retrain without leakage ───────────────────────────────────────────────
X2 = df[CLEAN_FEATURES].fillna(0)
y2 = df['suspicious']

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

model2 = XGBClassifier(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    scale_pos_weight = 2.52,
    random_state     = 42,
    eval_metric      = 'logloss',
    use_label_encoder= False,
    n_jobs           = -1,
    tree_method      = 'hist',
)

model2.fit(
    X_train2, y_train2,
    eval_set = [(X_test2, y_test2)],
    verbose  = 50,
)

# ── Evaluate ──────────────────────────────────────────────────────────────
y_prob2 = model2.predict_proba(X_test2)[:, 1]
prec, rec, thresholds = precision_recall_curve(y_test2, y_prob2)
f1s      = 2 * prec * rec / (prec + rec + 1e-9)
best_thr2 = thresholds[np.argmax(f1s)]
y_pred2   = (y_prob2 >= best_thr2).astype(int)

print(f"\nOptimal Threshold : {best_thr2:.4f}")
print(f"ROC-AUC           : {roc_auc_score(y_test2, y_prob2):.4f}")
print("\nClassification Report:")
print(classification_report(y_test2, y_pred2,
      target_names=['Legitimate', 'Suspicious']))

Yeh features label banane mein use hue — model inhe hata ke train karo
Leakage features : 13

Clean features   : 25
[0]	validation_0-logloss:0.65544
[50]	validation_0-logloss:0.15651
[100]	validation_0-logloss:0.09415
[150]	validation_0-logloss:0.07672
[200]	validation_0-logloss:0.06899
[250]	validation_0-logloss:0.06492
[299]	validation_0-logloss:0.06255

Optimal Threshold : 0.5171
ROC-AUC           : 0.9935

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.98      1.00      0.99    204651
  Suspicious       0.99      0.96      0.97     81258

    accuracy                           0.99    285909
   macro avg       0.99      0.98      0.98    285909
weighted avg       0.99      0.99      0.99    285909



In [19]:
import joblib

# ── Model save karo ───────────────────────────────────────────────────────
joblib.dump({
    'model'        : model2,
    'features'     : CLEAN_FEATURES,
    'threshold'    : best_thr2,
    'label_encoder': le,
    'p95_changes'  : p95,
}, "github_suspicious_model_final.pkl")

print("✅ Model saved → github_suspicious_model_final.pkl")

# ── SHAP — kaunse features important hain ─────────────────────────────────
import shap

explainer   = shap.TreeExplainer(model2)
shap_values = explainer.shap_values(X_test2.sample(5000, random_state=42))

shap.summary_plot(shap_values,
                  X_test2.sample(5000, random_state=42),
                  feature_names=CLEAN_FEATURES,
                  show=True)

✅ Model saved → github_suspicious_model_final.pkl
